# Power System Load Type Classification

**Objective:** predict `Light_Load`, `Medium_Load`, or `Maximum_Load` from historical power-system readings.

**Validation design:** sort by timestamp and reserve the latest calendar month as the final test set. This avoids the optimistic evaluation that can occur when time-dependent data is randomly split.

Run from the project root after adding `data/load_data.csv`.


## 1. Imports and reproducibility

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score)
from IPython.display import display

RANDOM_STATE = 42
DATA_PATH = Path("data/load_data.csv")
FIG_DIR = Path("figures")
ART_DIR = Path("artifacts")
FIG_DIR.mkdir(exist_ok=True)
ART_DIR.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid")


## 2. Load and validate the dataset
If the CSV is missing, add it to `data/` before continuing. We deliberately do not create fake observations or fake scores.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Put the assignment CSV at {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)
df.columns = [str(c).strip() for c in df.columns]
if "Date_Time" not in df.columns:
    candidates = [c for c in ["Date", "Datetime", "Timestamp", "date", "datetime"] if c in df.columns]
    if candidates:
        df = df.rename(columns={candidates[0]: "Date_Time"})
if "Load_Type" not in df.columns:
    candidates = [c for c in df.columns if c.lower().replace("_", "").replace(" ", "") == "loadtype"]
    if candidates:
        df = df.rename(columns={candidates[0]: "Load_Type"})
required = {"Date_Time", "Load_Type"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}. Found: {list(df.columns)}")

df["Date_Time"] = pd.to_datetime(df["Date_Time"], dayfirst=True, errors="coerce")
before = len(df)
df = df.dropna(subset=["Date_Time", "Load_Type"]).copy()
df["Load_Type"] = df["Load_Type"].astype(str).str.strip()
df = df[df["Load_Type"] != ""].sort_values("Date_Time").drop_duplicates().reset_index(drop=True)
print(f"Rows retained: {len(df):,}; removed: {before-len(df):,}")
print(f"Date range: {df.Date_Time.min()} to {df.Date_Time.max()}")
display(df.head())
display(df.isna().sum().rename("missing_values").to_frame())


## 3. Exploratory data analysis (EDA)
EDA helps assess class imbalance, changes over time, numeric relationships, and whether classes differ in energy usage.

In [ ]:
class_counts = df["Load_Type"].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
class_counts.plot(kind="bar", ax=ax)
ax.set(title="Load type class distribution", xlabel="Load type", ylabel="Number of records")
plt.xticks(rotation=20); plt.tight_layout()
plt.savefig(FIG_DIR / "01_class_distribution.png", dpi=160)
plt.show()


In [ ]:
if "Usage_kWh" in df.columns:
    fig, ax = plt.subplots(figsize=(12, 4))
    for label, part in df.groupby("Load_Type"):
        ax.scatter(part["Date_Time"], part["Usage_kWh"], s=8, alpha=.5, label=label)
    ax.set(title="Energy usage over time by load type", xlabel="Date", ylabel="Usage (kWh)")
    ax.legend(); plt.tight_layout()
    plt.savefig(FIG_DIR / "02_usage_over_time.png", dpi=160)
    plt.show()


In [ ]:
numeric_df = df.select_dtypes(include=np.number)
if numeric_df.shape[1] >= 2:
    fig, ax = plt.subplots(figsize=(10, 7))
    sns.heatmap(numeric_df.corr(), cmap="coolwarm", center=0, annot=numeric_df.shape[1] <= 10, fmt=".2f", ax=ax)
    ax.set_title("Numeric feature correlation")
    plt.tight_layout(); plt.savefig(FIG_DIR / "03_correlation_heatmap.png", dpi=160); plt.show()


## 4. Feature engineering
Extract hour/day/weekday/month and use sine/cosine encodings for cyclical time. This lets the model represent that 23:00 is close to 00:00.

In [ ]:
dt = df["Date_Time"]
work = df.copy()
work["hour"] = dt.dt.hour + dt.dt.minute / 60
work["day"] = dt.dt.day
work["weekday"] = dt.dt.dayofweek
work["month"] = dt.dt.month
work["dayofyear"] = dt.dt.dayofyear
work["hour_sin"] = np.sin(2*np.pi*work["hour"]/24)
work["hour_cos"] = np.cos(2*np.pi*work["hour"]/24)
work["weekday_sin"] = np.sin(2*np.pi*work["weekday"]/7)
work["weekday_cos"] = np.cos(2*np.pi*work["weekday"]/7)
display(work.head())


## 5. Chronological train/test split
All records in the latest month become test data; every earlier month is training data. The holdout month is not used to fit the model or imputation values.

In [ ]:
last_month = work["Date_Time"].dt.to_period("M").max()
test_mask = work["Date_Time"].dt.to_period("M").eq(last_month)
feature_cols = [c for c in work.columns if c not in ["Date_Time", "Load_Type"]]
X = work[feature_cols].replace([np.inf, -np.inf], np.nan)
y = work["Load_Type"]
X_train, X_test = X.loc[~test_mask].copy(), X.loc[test_mask].copy()
y_train, y_test = y.loc[~test_mask], y.loc[test_mask]
if X_train.empty or X_test.empty:
    raise ValueError("Need at least two calendar months for the chronological holdout.")
medians = X_train.median(numeric_only=True)
X_train = X_train.fillna(medians).fillna(0)
X_test = X_test.fillna(medians).fillna(0)
print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
print(f"Test month: {last_month}")
print(f"Training period: {work.loc[~test_mask, 'Date_Time'].min()} to {work.loc[~test_mask, 'Date_Time'].max()}")


## 6. Train Extra Trees classifier
Tree ensembles can capture nonlinear relationships and usually do not require feature scaling. A fixed random seed makes the run reproducible.

In [ ]:
model = ExtraTreesClassifier(
    n_estimators=500,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)
model.fit(X_train, y_train)
pred = model.predict(X_test)


## 7. Evaluation
Accuracy is reported alongside balanced accuracy, macro precision/recall/F1 and per-class metrics. This is important if classes are imbalanced.

In [ ]:
labels = sorted(set(y_train) | set(y_test))
metrics = {
    "test_month": str(last_month),
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "accuracy": float(accuracy_score(y_test, pred)),
    "balanced_accuracy": float(balanced_accuracy_score(y_test, pred)),
    "macro_precision": float(precision_score(y_test, pred, average="macro", zero_division=0)),
    "macro_recall": float(recall_score(y_test, pred, average="macro", zero_division=0)),
    "macro_f1": float(f1_score(y_test, pred, average="macro", zero_division=0)),
}
print(json.dumps(metrics, indent=2))
print(classification_report(y_test, pred, labels=labels, zero_division=0))


In [ ]:
cm = confusion_matrix(y_test, pred, labels=labels)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=ax)
ax.set(title="Confusion matrix — latest month", xlabel="Predicted", ylabel="Actual")
plt.tight_layout(); plt.savefig(FIG_DIR / "07_confusion_matrix.png", dpi=160); plt.show()


In [ ]:
importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values().tail(20)
fig, ax = plt.subplots(figsize=(9, 6))
importance.plot(kind="barh", ax=ax)
ax.set(title="Top feature importances", xlabel="Importance")
plt.tight_layout(); plt.savefig(FIG_DIR / "06_feature_importance.png", dpi=160); plt.show()


## 8. Save reproducible outputs
The model bundle contains the feature order and training medians, so later prediction uses the same inputs. Metrics and row-level predictions are saved for reporting.

In [ ]:
import joblib
joblib.dump(
    {"model": model, "feature_columns": feature_cols, "medians": medians.to_dict(), "test_month": str(last_month)},
    ART_DIR / "load_type_model.joblib"
)
with open(ART_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump({**metrics, "classification_report": classification_report(
        y_test, pred, labels=labels, output_dict=True, zero_division=0),
        "labels": labels, "confusion_matrix": cm.tolist(), "features": feature_cols}, f, indent=2)
pd.DataFrame({
    "Date_Time": work.loc[y_test.index, "Date_Time"].values,
    "Actual_Load_Type": y_test.values,
    "Predicted_Load_Type": pred
}).to_csv(ART_DIR / "last_month_predictions.csv", index=False)
print("Saved model, metrics, predictions and figures.")


## 9. Critical interpretation and limitations
- A strong score on one holdout month is not proof that the model will perform equally well in every season.
- Inspect per-class recall: a low value means the model misses that load category.
- Feature importance describes model reliance, not causal influence.
- For stronger validation, run rolling-origin tests on several different months and tune hyperparameters only within each training period.
- If the data has very few months, the holdout score will be uncertain. Always describe the actual test month and dataset size in the submission.
